In [ ]:
import glob, os, shutil, subprocess, tempfile
from pathlib import Path
OCTAVE = shutil.which("octave-cli") or shutil.which("octave")
if OCTAVE is None and os.name == "nt":
    candidates = glob.glob(r"C:\Program Files\GNU Octave\Octave-*\mingw64\bin\octave-cli.exe")
    if candidates:
        OCTAVE = sorted(candidates)[-1]
if OCTAVE is None:
    raise RuntimeError("Octave CLI not found. Add octave-cli to PATH or install GNU Octave.")
WORK = Path(tempfile.mkdtemp(prefix="me256-coursework-"))
print(f"Octave: {OCTAVE}")
print(f"Temporary source directory: {WORK}")

for relative in ['problem-1', 'river-fence']:
    (WORK / relative).mkdir(parents=True, exist_ok=True)


# ME256 Homework 1: constrained optimization and a minimum-fence problem

**Official statement:** [ME256 Homework 1 (2011)](https://mecheng.iisc.ac.in/suresh/me256/hw2011/homework1.html). The outline below is a study summary in my own words; consult the linked page for the complete statement and figure.

## Problem 1 — constrained reciprocal objective

Choose nonnegative design variables $x_1,x_2$ to minimize

$$f(x_1,x_2)=\frac{5}{x_1}+\frac{2}{x_2}$$

subject to $x_1+x_2\leq 3$, $4x_1+x_2\leq 6$, and $x_1+3x_2\leq 20$. The archived script plots the constraint boundaries, feasible region, and objective contours, and compares `fmincon` runs from several initial points. The assignment also asks for a hand solution and a discussion of how the solver approaches the optimum.

## Problem 2 — shortest farm boundary

Model the farm boundary separating a patch from a river as a curve whose length is to be minimized, while keeping the enclosed area fixed. The archived discretization uses a 10 m horizontal span and a straight river boundary described by $y=8-0.8x$. The statement asks students to compare prescribed area cases of 25 m$^2$ and 35 m$^2$; a parenthetical in the online page repeats 25 m$^2$ for the second case, so check the page/figure when reproducing that comparison. The script currently uses one discretization and the 25 m$^2$ case.

## What to revisit

- Translate verbal limits and geometry into linear/nonlinear constraints.
- Compare a hand analysis with a numerical constrained optimizer and interpret solver paths from different initial guesses.
- Understand how the continuous boundary is represented by discrete variables and how area constraints affect the design.

## Reproducibility

This notebook uses a Python 3 Jupyter kernel to materialize the embedded MATLAB source into a temporary directory, then invokes Octave CLI. The `.m` sources are embedded below; they are not stored separately in this repository. Install Jupyter/IPython and an Octave installation with the `optim` package. MATLAB with Optimization Toolbox remains the closest match to the original runtime.

The first script plots the constraints and objective contours, then invokes `fmincon` repeatedly with different initial points. Compare the plotted paths and returned optimizer values; the callbacks in `outfun.m`, `outfun1.m`, and `outfun2.m` label iterations with different markers.

The second script discretizes the boundary and passes its objective and area constraint to `fmincon`. The original files use a particular set of geometric assumptions and parameter values; consult the official statement before interpreting the plot or changing them. Possible alternative discretizations or formulations belong in a separate study note, not as silent changes to this transcription.

## Embedded source code

Each following code cell contains one MATLAB source file and writes it into a temporary working directory. Run the notebook from top to bottom; no reconstructed `.m` files are required in the repository.

### `problem-1/HW1_1_1.m`

In [ ]:
%%writefile {WORK / 'problem-1' / 'HW1_1_1.m'}
% Assignment 1, problem 1: constrained reciprocal-objective optimization.
% The objective and inequality constraints are transcribed from codes.pdf.
clc;
clear all;
close all;

f = @(x, y) 5 * x^-1 + 2 * y^-1; % Objective shown in the contour plot.
g1 = @(x, y) x + y - 3;
g2 = @(x, y) 4 * x + y - 6;
g3 = @(x, y) x + 3 * y - 20;
g4 = @(x, y) -x;
g5 = @(x, y) -y;

ezplot(g1, [0, 10, 0, 10]);
hold on;
ezplot(g2, [0, 10, 0, 10]);
hold on;
ezplot(g3, [0, 10, 0, 10]);
hold on;
ezplot(g4, [0, 10, 0, 10]);
hold on;
ezplot(g5, [0, 10, 0, 10]);
hold on;
area([0; 1; 1.5], [3; 2; 0]);
hold on;
ezcontour(f, [0, 10, 0, 10], 100);
shading interp;
colormap copper;
hold on;

linear_constraints = [1, 1; 4, 1; 1, 3; -1, 0; 0, -1];
constraint_bounds = [3; 6; 20; 0; 0];

% Keep the optimizer's result available for inspection in the workspace.
[x, fval] = fmincon(@tran, [4; 1], linear_constraints, constraint_bounds, ...
    [], [], [], [], [], optimset('Algorithm', 'active-set'));
fprintf('Start [4, 1]: x = [%.6f, %.6f], f = %.6f\n', x(1), x(2), fval);
plot(x(1), x(2), 'o');
hold on;
if ~exist('OCTAVE_VERSION', 'builtin')
    legend('x1+x2-3<=0', '4*x1+x2-6<=0', 'x1+3*x2-20<=0', ...
        '-x1<=0', '-x2<=0', 'feasible region', ...
        'f=5/x1+2/x2', 'optimum occurs here');
end
xlabel('x1');
ylabel('x2');
title('Problem no. 1');
hold on;

% Compare the iteration paths from the three starting points in the PDF.
[x, fval] = fmincon(@tran, [4; 1], linear_constraints, constraint_bounds, ...
    [], [], [], [], [], optimset('Algorithm', 'active-set', ...
    'OutputFcn', @outfun));
fprintf('Start [4, 1] with iteration plot: x = [%.6f, %.6f], f = %.6f\n', ...
    x(1), x(2), fval);
hold on;
[x, fval] = fmincon(@tran, [1; 4], linear_constraints, constraint_bounds, ...
    [], [], [], [], [], optimset('Algorithm', 'active-set', ...
    'OutputFcn', @outfun1));
fprintf('Start [1, 4]: x = [%.6f, %.6f], f = %.6f\n', x(1), x(2), fval);
hold on;
[x, fval] = fmincon(@tran, [4; 4], linear_constraints, constraint_bounds, ...
    [], [], [], [], [], optimset('Algorithm', 'active-set', ...
    'OutputFcn', @outfun2));
fprintf('Start [4, 4]: x = [%.6f, %.6f], f = %.6f\n', x(1), x(2), fval);
hold off;


### `problem-1/tran.m`

In [ ]:
%%writefile {WORK / 'problem-1' / 'tran.m'}
function f = tran(x)
% Objective function for Assignment 1, problem 1.
f = 5 / x(1) + 2 / x(2);
end


### `problem-1/outfun.m`

In [ ]:
%%writefile {WORK / 'problem-1' / 'outfun.m'}
function stop = outfun(x, optimValues, state)
% Plot '+' markers for iterates from the first initial point.
stop = false;

switch state
    case 'init'
        hold on;
    case 'iter'
        plot(x(1), x(2), '+');
        text(x(1) + 0.15, x(2), num2str(optimValues.iteration));
    case 'done'
        hold off;
end
end


### `problem-1/outfun1.m`

In [ ]:
%%writefile {WORK / 'problem-1' / 'outfun1.m'}
function stop = outfun1(x, optimValues, state)
% Plot square markers for iterates from the second initial point.
stop = false;

switch state
    case 'init'
        hold on;
    case 'iter'
        plot(x(1), x(2), 's');
        text(x(1) + 0.15, x(2), num2str(optimValues.iteration));
    case 'done'
        hold off;
end
end


### `problem-1/outfun2.m`

In [ ]:
%%writefile {WORK / 'problem-1' / 'outfun2.m'}
function stop = outfun2(x, optimValues, state)
% Plot '*' markers for iterates from the third initial point.
stop = false;

switch state
    case 'init'
        hold on;
    case 'iter'
        plot(x(1), x(2), '*');
        text(x(1) + 0.15, x(2), num2str(optimValues.iteration));
    case 'done'
        hold off;
end
end


### `river-fence/river_fence.m`

In [ ]:
%%writefile {WORK / 'river-fence' / 'river_fence.m'}
% Assignment 1 river/fence problem: minimize fence length subject to area
% and geometric constraints. The formulation follows the original PDF.
clc;
clear all;
close all;

global n Area
Area = 25;
ymax = 8;
xmaxx = 10;
xminn = 0;
n = 50;
slope = ymax / (xmaxx - xminn);

lb = [zeros(1, n + 1), 0, 0];
y0 = [ones(n + 1, 1); xmaxx - 2; xminn + 2];
A = [eye(n + 1), [0:slope / n:slope]', [slope:-slope / n:0]'; ...
    zeros(2, n + 1), eye(2)];
b = [ones(n + 1, 1) * ymax; xmaxx; xmaxx];

[y, fval, exitflag, output] = fmincon(@river1, y0, A, b, ...
    [zeros(1, n), 1, 0, 0], [0], ...
    lb, [], @area, optimset('Algorithm', 'active-set'));
fprintf('Fence objective %.6f; exit flag %d; iterations %d; constraint violation %.6g\n', ...
    fval, exitflag, output.niter, output.constrviolation);

plot([y(n + 3):(y(n + 2) - y(n + 3)) / n:y(n + 2)]', y(1:n + 1));
hold on;
plot([0, xmaxx], [ymax, 0], 'r');
legend('Fence', 'River Boundary');
xlabel('x');
ylabel('y');
title('Shape of the Fence');
hold off;


### `river-fence/river1.m`

In [ ]:
%%writefile {WORK / 'river-fence' / 'river1.m'}
function f = river1(y)
% Fence-length objective from the original discretized formulation.
global n
f = 0;
for k = 1:n
    f = f + sqrt(((y(n + 2) - y(n + 3)) / n)^2 + ...
        (y(k + 1) - y(k))^2);
end
end


### `river-fence/area.m`

In [ ]:
%%writefile {WORK / 'river-fence' / 'area.m'}
function [c, ceq] = area(y)
% Nonlinear equality constraints for the river/fence problem.
global n Area
ceq(1) = (y(1) + y(n + 1) + 2 * sum(y(2:n))) * ...
    (y(n + 2) - y(n + 3)) - 2 * Area * n;
ceq(2) = y(1) * y(n + 3);
c = [];
end


## Octave 11.3.0 test results

The reciprocal-objective script completed from all three initial points, returning approximately $x=(1.139620,1.441518)$ and $f=5.774852$. The river-fence script runs, but Octave `optim` reports exit flag 0 after its 100-iteration limit and a constraint violation of about 0.119; treat its plot/result as **not converged**, not as a verified solution. The reciprocal contour has singularities at zero, which produce nonfatal plotting warnings.

The full command output is retained in the executed cell below. These are Octave results; compare with MATLAB/Optimization Toolbox before treating them as the historical result.

In [ ]:
import subprocess

def run_case(label, folder, script, needs_optim=True):
    setup = "set(0,'defaultfigurevisible','off');"
    if needs_optim:
        setup += "pkg load optim;"
    setup += "if any(strcmp(available_graphics_toolkits(),'gnuplot')), graphics_toolkit('gnuplot'); end;"
    setup += f"run('{script}');"
    result = subprocess.run(
        [OCTAVE, "--no-gui", "--quiet", "--eval", setup],
        cwd=WORK / folder, text=True, capture_output=True, errors="replace"
    )
    status = "PASS" if result.returncode == 0 else "FAILED"
    print(f"\n=== {label}: {status} (exit code {result.returncode}) ===")
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    return result.returncode

run_case('Problem 1 - reciprocal objective', 'problem-1', 'HW1_1_1.m', True)
run_case('Problem 2 - river fence', 'river-fence', 'river_fence.m', True)



=== Problem 1 - reciprocal objective: PASS (exit code 0) ===
Start [4, 1]: x = [1.139620, 1.441518], f = 5.774852
Start [4, 1] with iteration plot: x = [1.139620, 1.441518], f = 5.774852
Start [1, 4]: x = [1.139620, 1.441518], f = 5.774852
Start [4, 4]: x = [1.139620, 1.441518], f = 5.774852
--- stderr ---

The gnuplot graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.  Communication with gnuplot
uses a one-directional pipe and limited information is passed back to the
Octave interpreter so most changes made interactively in the plot window
will not be reflected in the graphics properties managed by Octave.  For
example, if the plot window is closed with a mouse click, Octave will not
be notified and will not update its internal list of open figure windows.
The qt toolkit is recommended instead.

The FLTK graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.
The qt toolkit is r

In [ ]:
import shutil
shutil.rmtree(WORK)
print("Temporary generated source files removed.")